# Time-Based Exit Strategy (4-Bar Exit) on SPY
## Strategy Brief
The Time-Based Exit Strategy, also known as the 4-Bar Exit, is a simple trading strategy that involves entering a trade based on a specific signal and exiting after a fixed number of bars, regardless of the price action. In this case, the strategy will be applied to SPY, the ETF tracking the S&P 500 index. The strategy aims to capture short-term price movements by holding a position for exactly four trading days. The results will be compared against a buy-and-hold strategy to evaluate its performance.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters for our strategy, such as the number of bars to hold a position and the financial instrument we are trading.

In [ ]:
TICKER = 'SPY'
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
HOLD_PERIOD = 4

## PHASE 2 - Data Exploration
We will download historical price data for SPY from Yahoo Finance and compute any necessary indicators. The data will be visualized to understand the price movements and the context for our strategy.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(TICKER, start=START_DATE, end=END_DATE)

# Plot the closing price
data['Close'].plot(title='SPY Closing Price', figsize=(14, 7))
plt.show()

## PHASE 3 - Strategy Engineering
The strategy involves identifying a signal to enter a trade and holding the position for a fixed period of 4 bars. We will create a signal series and define the entry and exit logic.

In [ ]:
# Simple signal: Buy if the previous day's close is higher than the close 2 days ago
signal = (data['Close'] > data['Close'].shift(2)).astype(int)

# Entry and Exit logic
positions = signal.copy()
positions = positions.shift(1).fillna(0)  # Enter on the next day

# Exit after 4 bars
for i in range(len(positions) - HOLD_PERIOD):
    if positions.iloc[i] == 1:
        positions.iloc[i+1:i+HOLD_PERIOD] = 0

## PHASE 4 - Coding & Backtesting
We will backtest the strategy by calculating the daily returns based on the positions and plotting the equity curve.

In [ ]:
# Calculate daily returns
data['Returns'] = data['Close'].pct_change()

# Calculate strategy returns
strategy_returns = positions.shift(1) * data['Returns']

# Calculate equity curve
equity_curve = (1 + strategy_returns).cumprod()

# Plot equity curve
equity_curve.plot(title='Equity Curve', figsize=(14, 7))
plt.show()

## PHASE 5 - Performance Evaluation
We will evaluate the performance of the strategy using metrics such as CAGR, Sharpe Ratio, Sortino Ratio, Calmar Ratio, and maximum drawdown. The results will be compared to a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(returns):
    cagr = (equity_curve[-1] ** (252.0 / len(returns))) - 1
    sharpe_ratio = np.sqrt(252) * returns.mean() / returns.std()
    downside_returns = returns[returns < 0]
    sortino_ratio = np.sqrt(252) * returns.mean() / downside_returns.std()
    max_drawdown = (equity_curve.cummax() - equity_curve).max()
    calmar_ratio = cagr / max_drawdown
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

strategy_metrics = calculate_performance_metrics(strategy_returns)
buy_and_hold_metrics = calculate_performance_metrics(data['Returns'])

# Display performance comparison
df_performance = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': strategy_metrics,
    'Buy & Hold': buy_and_hold_metrics
})
print(df_performance)

## PHASE 6 - Deploy & Monitor
We will create a function that downloads the last 60 days of SPY data, computes the current signal, and prints the position for today.

In [ ]:
def get_current_signal():
    recent_data = yf.download(TICKER, period='60d')
    recent_signal = (recent_data['Close'] > recent_data['Close'].shift(2)).astype(int)
    current_position = recent_signal.iloc[-1]
    print(f"Current position for {TICKER}: {'Long' if current_position == 1 else 'No Position'}")

get_current_signal()